# Rearc Data Quest - Analysis

This notebook performs:

1. Population statistics (2013–2018)
2. Best year per series
3. Join between BLS dataset and population data

In [22]:
import pandas as pd
import boto3
import json
from io import BytesIO

In [23]:
s3 = boto3.client("s3")

BUCKET = "rearc-data-quest-karan"

In [24]:
obj = s3.get_object(
    Bucket=BUCKET,
    Key="population/population.json"
)

data = json.loads(obj["Body"].read())

pop_df = pd.DataFrame(data["data"])

pop_df.head()

,Nation ID,Nation,Year,Population
0,01000US,United States,2013,316128839.0
1,01000US,United States,2014,318857056.0
2,01000US,United States,2015,321418821.0
3,01000US,United States,2016,323127515.0
4,01000US,United States,2017,325719178.0


In [25]:
obj = s3.get_object(
    Bucket=BUCKET,
    Key="bls/pr.data.0.Current"
)

bls = pd.read_csv(
    BytesIO(obj["Body"].read()),
    sep="\t"
)

bls.columns = bls.columns.str.strip()

bls["series_id"] = bls["series_id"].str.strip()
bls["period"] = bls["period"].str.strip()

bls.head()

,series_id,year,period,value,footnote_codes
0,PRS30006011,1995,Q01,2.6,NaN
1,PRS30006011,1995,Q02,2.1,NaN
2,PRS30006011,1995,Q03,0.9,NaN
3,PRS30006011,1995,Q04,0.1,NaN
4,PRS30006011,1995,Q05,1.4,NaN


In [26]:
df = pop_df[
    (pop_df["Year"] >= 2013) &
    (pop_df["Year"] <= 2018)
]

mean_population = df["Population"].mean()
std_population = df["Population"].std()

print("Mean : ", mean_population, "Standard Deviation : ", std_population)

Mean :  322069808.0 Standard Deviation :  4158441.040908095


In [27]:
result = (
    bls.groupby(["series_id", "year"])["value"]
    .sum()
    .reset_index()
)

best = (
    result.sort_values(["series_id", "value"], ascending=[True, False])
    .groupby("series_id")
    .head(1)
)

best.head()

,series_id,year,value
27,PRS30006011,2022,20.500
58,PRS30006012,2022,17.100
65,PRS30006013,1998,705.895
108,PRS30006021,2010,17.700
139,PRS30006022,2010,12.400


In [28]:
filtered = bls[
    (bls["series_id"] == "PRS30006032") &
    (bls["period"] == "Q01")
]

joined = filtered.merge(
    pop_df,
    left_on="year",
    right_on="Year",
    how="inner"
)

joined[["series_id", "year", "period", "value", "Population"]].head()

,series_id,year,period,value,Population
0,PRS30006032,2013,Q01,0.5,316128839.0
1,PRS30006032,2014,Q01,-0.1,318857056.0
2,PRS30006032,2015,Q01,-1.7,321418821.0
3,PRS30006032,2016,Q01,-1.4,323127515.0
4,PRS30006032,2017,Q01,0.9,325719178.0
